In [ ]:
import os, subprocess, sys, json, time
os.environ["PYTHONUTF8"]="1"
print(subprocess.run([sys.executable,"-m","pip","install","-q","flopscope>=0.12.1,<0.13","whestbench>=0.16.1,<0.17"],capture_output=True,text=True).stdout[-500:])
import flopscope; from importlib.metadata import version; print("flopscope", version("flopscope"), "whestbench", version("whestbench"))
import os; print("cpus", os.cpu_count()); print(subprocess.run(["free","-g"],capture_output=True,text=True).stdout)

In [ ]:
import glob
hits = glob.glob("/kaggle/input/**/dump_driver.py", recursive=True); print("driver at", hits)
INPUT = os.path.dirname(hits[0])


In [ ]:
# Download only the mini split (prepared Arrow fast path) into the HF cache
os.environ["HF_XET_HIGH_PERFORMANCE"]="1"
t0=time.time()
r=subprocess.run(["whest","dataset","download","aicrowd/arc-whestbench-public-2026","--revision","v2-phase2","--split","mini"],capture_output=True,text=True)
print(r.stdout[-800:], r.stderr[-800:]); print("download s", time.time()-t0)

In [ ]:
env = dict(os.environ); env.update({"EST": f"{INPUT}/v33dump.py", "DUMP_DIR": "/kaggle/working/dumps", "START": "50", "END": "100", "PYTHONWARNINGS": "ignore", "PYTHONUTF8": "1"})
t0 = time.time()
p = subprocess.Popen([sys.executable, f"{INPUT}/dump_driver.py"], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "Warning" in line or "warn" in line: continue
    print(line, end="", flush=True)
p.wait(); print("exit", p.returncode, "elapsed", time.time() - t0)
print(subprocess.run(["du", "-sh", "/kaggle/working/dumps"], capture_output=True, text=True).stdout)
